# 1. 라이브러리&데이터 불러오기

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, classification_report

path = '/content/drive/MyDrive/Colab Notebooks/car.data'

columns = [
    'buying', 'maint', 'doors',
    'persons', 'lug_boot', 'safety', 'class'
]

df = pd.read_csv(path, header=None, names=columns)

print(df.head())
print(df.info())
print(df['class'].value_counts())


  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   buying    1728 non-null   object
 1   maint     1728 non-null   object
 2   doors     1728 non-null   object
 3   persons   1728 non-null   object
 4   lug_boot  1728 non-null   object
 5   safety    1728 non-null   object
 6   class     1728 non-null   object
dtypes: object(7)
memory usage: 94.6+ KB
None
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


# 2. 입력 데이터(X)와 정답(y) 분리

In [4]:
X = df.drop('class', axis=1)
y = df['class']


# 3. 학습 데이터와 테스트 데이터 분리

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 4. 범주형 데이터 인코딩

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), X.columns)
    ]
)


# 5. 분류 모델 정의

In [7]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}


# 6. 모델 학습 및 평가

In [8]:
results = {}

for name, model in models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results[name] = accuracy

    print(f'\n===== {name} =====')
    print(f'Accuracy: {accuracy:.4f}')
    print(classification_report(y_test, y_pred, zero_division=0))



===== Decision Tree =====
Accuracy: 0.9740
              precision    recall  f1-score   support

         acc       0.96      0.92      0.94        77
        good       0.88      1.00      0.93        14
       unacc       0.98      1.00      0.99       242
       vgood       1.00      0.85      0.92        13

    accuracy                           0.97       346
   macro avg       0.95      0.94      0.95       346
weighted avg       0.97      0.97      0.97       346


===== Random Forest =====
Accuracy: 0.9682
              precision    recall  f1-score   support

         acc       0.94      0.96      0.95        77
        good       0.81      0.93      0.87        14
       unacc       0.99      0.99      0.99       242
       vgood       0.90      0.69      0.78        13

    accuracy                           0.97       346
   macro avg       0.91      0.89      0.90       346
weighted avg       0.97      0.97      0.97       346


===== Logistic Regression =====
Accuracy:

# 7. 모델별 정확도 비교

In [9]:
results_df = pd.DataFrame(
    results.items(),
    columns=['Model', 'Accuracy']
)

print('\n===== 모델별 정확도 비교 =====')
print(results_df.sort_values(by='Accuracy', ascending=False))


===== 모델별 정확도 비교 =====
                 Model  Accuracy
0        Decision Tree  0.973988
1        Random Forest  0.968208
2  Logistic Regression  0.901734
3                  KNN  0.893064
